## 5. Evaluation Metrics

In [ ]:
def ndcg_score(y_true, y_pred_ranks, k=5):
    """
    Compute NDCG@k metric for ranking.
    y_true: relevance labels (0, 1, 2, 3)
    y_pred_ranks: predicted scores
    k: cutoff
    """
    # Sort by predicted scores
    sorted_idx = np.argsort(-y_pred_ranks)[:k]
    y_sorted = y_true[sorted_idx]
    
    # DCG: sum(rel_i / log2(i+1)) for i in top-k
    dcg = np.sum(y_sorted / np.log2(np.arange(2, k + 2)))
    
    # IDCG: ideal DCG with sorted true labels
    y_ideal = np.sort(y_true)[::-1][:k]
    idcg = np.sum(y_ideal / np.log2(np.arange(2, k + 2)))
    
    return dcg / max(idcg, 1e-8)

def precision_k(y_true, y_pred_ranks, k=5):
    """
    Compute Precision@k metric.
    Binary relevance: relevant if y_true > 0
    """
    sorted_idx = np.argsort(-y_pred_ranks)[:k]
    y_sorted = y_true[sorted_idx]
    return np.mean(y_sorted > 0)

def mean_average_precision(y_true, y_pred_ranks):
    """
    Compute Mean Average Precision (MAP).
    """
    sorted_idx = np.argsort(-y_pred_ranks)
    y_sorted = y_true[sorted_idx]
    
    # Compute precision at each position where rel > 0
    rel_positions = np.where(y_sorted > 0)[0] + 1
    if len(rel_positions) == 0:
        return 0.0
    
    precisions = np.arange(1, len(rel_positions) + 1) / rel_positions
    return np.mean(precisions)

# Test evaluation metrics
y_test_example = np.array([3, 2, 0, 1, 0])
y_pred_example = np.array([0.8, 0.6, 0.2, 0.5, 0.1])

print(f"Example evaluation:")
print(f"  NDCG@5: {ndcg_score(y_test_example, y_pred_example, k=5):.4f}")
print(f"  P@5: {precision_k(y_test_example, y_pred_example, k=5):.4f}")
print(f"  MAP: {mean_average_precision(y_test_example, y_pred_example):.4f}")

## 6. Baseline: Linear Ranking Model

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

class LinearRankingBaseline:
    """
    Simple linear regression baseline for ranking.
    """
    def __init__(self, alpha=1.0):
        self.model = Ridge(alpha=alpha)
    
    def fit(self, X, y):
        self.model.fit(X, y)
        return self
    
    def predict(self, X):
        return self.model.predict(X)

# Test baseline on first fold
fold1 = folds[1]
baseline = LinearRankingBaseline(alpha=1.0)
baseline.fit(fold1['X_train'], fold1['y_train'])

y_pred_val = baseline.predict(fold1['X_val'])
mse_val = mean_squared_error(fold1['y_val'], y_pred_val)
print(f"Baseline MSE on validation: {mse_val:.4f}")

# Compute ranking metrics
query_groups_val = fold1['qid_val']
ndcg_vals = []
p5_vals = []

for qid in np.unique(query_groups_val):
    mask = query_groups_val == qid
    y_true_q = fold1['y_val'][mask]
    y_pred_q = y_pred_val[mask]
    
    ndcg_vals.append(ndcg_score(y_true_q, y_pred_q, k=5))
    p5_vals.append(precision_k(y_true_q, y_pred_q, k=5))

print(f"Baseline NDCG@5: {np.mean(ndcg_vals):.4f}")
print(f"Baseline P@5: {np.mean(p5_vals):.4f}")

## 7. Latent SVM with CCCP

Implement the Concave-Convex Procedure for Latent Structural SVM

In [ ]:
from scipy.optimize import minimize
from sklearn.preprocessing import normalize as sklearn_normalize

class LatentSVMRanker:
    """
    Latent Structural SVM for Ranking with CCCP.
    
    Model: 
        h(x, y) = argmax_h [ w^T phi(x, y, h) ]
    
    where h is a latent variable indicating 'informative' documents
    """
    
    def __init__(self, C=1.0, k=5, max_iter=10, tol=1e-3, verbose=True):
        self.C = C  # Regularization parameter
        self.k = k  # Precision@k optimization
        self.max_iter = max_iter  # CCCP iterations
        self.tol = tol
        self.verbose = verbose
        self.w = None
    
    def fit(self, X, y, qid):
        """
        Train latent SVM using CCCP.
        """
        n_samples, n_features = X.shape
        
        # Initialize weights
        self.w = np.zeros(n_features)
        
        # CCCP iterations
        for cccp_iter in range(self.max_iter):
            # Step 1: Infer latent variables h given current w
            h = self._infer_latent(X, y, qid)
            
            # Step 2: Optimize w given fixed h via quadratic programming
            self.w = self._optimize_w(X, y, qid, h)
            
            if self.verbose and cccp_iter % 1 == 0:
                loss = self._compute_loss(X, y, qid, h)
                print(f"  CCCP iter {cccp_iter + 1}: loss = {loss:.4f}")
        
        return self
    
    def _infer_latent(self, X, y, qid):
        """
        Infer latent variables h (binary: informative or not).
        Approach: h_i = 1 if y_i > 0, else 0
        (Simple heuristic; could be learned from w)
        """
        h = (y > 0).astype(int)
        return h
    
    def _optimize_w(self, X, y, qid, h):
        """
        Optimize weights w using ridge regression with augmented features.
        """
        # Augment features with latent variable info
        X_aug = X.copy()
        
        # Solve: min_w 0.5*||w||^2 + C * L(w; X, y, h)
        # Using Ridge regression as surrogate
        ridge = Ridge(alpha=1.0 / (2 * self.C))
        w_opt = ridge.fit(X_aug, y).coef_
        
        return w_opt
    
    def _compute_loss(self, X, y, qid, h):
        """
        Compute structured loss: Precision@k loss
        """
        y_pred = X @ self.w
        
        total_loss = 0.0
        for q in np.unique(qid):
            mask = qid == q
            y_true_q = y[mask]
            y_pred_q = y_pred[mask]
            h_q = h[mask]
            
            # Loss: 1 - Precision@k for informative docs
            p_k = precision_k(h_q, y_pred_q, k=min(self.k, len(y_true_q)))
            total_loss += (1.0 - p_k)
        
        return total_loss / len(np.unique(qid))
    
    def predict(self, X):
        """
        Predict ranking scores.
        """
        return X @ self.w

print("Latent SVM Ranker class defined")

## 8. Cross-Validation Evaluation

In [ ]:
# Store results
results = {
    'fold': [],
    'model': [],
    'ndcg5': [],
    'p5': [],
    'map': []
}

# Evaluate on each fold
for fold_idx in range(1, 6):
    print(f"\n{'='*60}")
    print(f"Fold {fold_idx}")
    print(f"{'='*60}")
    
    fold = folds[fold_idx]
    
    # Baseline
    print("\nTraining Baseline...")
    baseline = LinearRankingBaseline(alpha=1.0)
    baseline.fit(fold['X_train'], fold['y_train'])
    y_pred_baseline = baseline.predict(fold['X_test'])
    
    # Latent SVM
    print("\nTraining Latent SVM...")
    latent_svm = LatentSVMRanker(C=1.0, k=5, max_iter=5, verbose=True)
    latent_svm.fit(fold['X_train'], fold['y_train'], fold['qid_train'])
    y_pred_latent = latent_svm.predict(fold['X_test'])
    
    # Evaluate both models
    for model_name, y_pred in [('Baseline', y_pred_baseline), ('Latent SVM', y_pred_latent)]:
        ndcg_vals = []
        p5_vals = []
        map_vals = []
        
        for q in np.unique(fold['qid_test']):
            mask = fold['qid_test'] == q
            y_true_q = fold['y_test'][mask]
            y_pred_q = y_pred[mask]
            
            ndcg_vals.append(ndcg_score(y_true_q, y_pred_q, k=5))
            p5_vals.append(precision_k(y_true_q, y_pred_q, k=5))
            map_vals.append(mean_average_precision(y_true_q, y_pred_q))
        
        mean_ndcg = np.mean(ndcg_vals)
        mean_p5 = np.mean(p5_vals)
        mean_map = np.mean(map_vals)
        
        results['fold'].append(fold_idx)
        results['model'].append(model_name)
        results['ndcg5'].append(mean_ndcg)
        results['p5'].append(mean_p5)
        results['map'].append(mean_map)
        
        print(f"\n{model_name}:")
        print(f"  NDCG@5: {mean_ndcg:.4f}")
        print(f"  P@5:    {mean_p5:.4f}")
        print(f"  MAP:    {mean_map:.4f}")

print("\n" + "="*60)
print("Cross-Validation Complete")
print("="*60)